# Advanced Problems with Solutions: Callable Class Attributes

This notebook builds from the idea that **class attributes may contain any Python object, including callables**.

It goes well beyond calling a plain function stored on a class. You will practice:

- inspecting class namespaces with `__dict__`
- understanding function binding through instances
- distinguishing plain functions, `staticmethod`, and `classmethod`
- storing callable objects as class attributes
- avoiding accidental method binding
- dynamically replacing callables
- inheritance and callable lookup
- callable registries and dispatch tables
- descriptors and `__get__`
- decorators that preserve callable metadata
- validation of plugin-style class APIs
- advanced debugging with `inspect`
- designing robust, extensible callable-based class APIs

All exercises include complete solutions and executable checks.

> Best practice: run the notebook from top to bottom. Most exercises are self-contained, but some explanatory cells build on earlier concepts.


## 0. Baseline: a function stored in a class namespace

A function defined inside a class body becomes an attribute of that class.

When accessed through the **class**, it behaves like the original function.

When accessed through an **instance**, Python's descriptor protocol may transform it into a bound method.


In [1]:
class Program:
    language = "Python"

    def say_hello():
        print(f"Hello from {Program.language}!")


print("Class dictionary contains:", "say_hello" in Program.__dict__)
print("Via class:", Program.say_hello)
print("Via getattr:", getattr(Program, "say_hello"))

Program.say_hello()
getattr(Program, "say_hello")()
Program.__dict__["say_hello"]()


Class dictionary contains: True
Via class: <function Program.say_hello at 0x0000021A5678D120>
Via getattr: <function Program.say_hello at 0x0000021A5678D120>
Hello from Python!
Hello from Python!
Hello from Python!


## 1. Problem — Predict class access vs instance access

Consider the class below:

```python
class Greeter:
    def hello():
        return "hello"
```

Without running the code first, predict what happens for:

1. `Greeter.hello`
2. `Greeter.hello()`
3. `Greeter().hello`
4. `Greeter().hello()`

Explain **why** the instance call behaves differently.


### Solution

Functions implement the descriptor protocol. Accessing the function through an instance invokes the function object's `__get__`, producing a bound method that automatically supplies the instance as the first positional argument.

But `hello` was defined with **zero parameters**, so calling the bound version causes Python to supply one argument that the function cannot accept.


In [2]:
class Greeter:
    def hello():
        return "hello"


g = Greeter()

print("1.", Greeter.hello)
print("2.", Greeter.hello())
print("3.", g.hello)

try:
    print("4.", g.hello())
except TypeError as exc:
    print("4. TypeError:", exc)


1. <function Greeter.hello at 0x0000021A5677D1C0>
2. hello
3. <bound method Greeter.hello of <__main__.Greeter object at 0x0000021A46708440>>
4. TypeError: Greeter.hello() takes 0 positional arguments but 1 was given


### Key lesson

A function inside a class is not merely "a callable sitting in a dictionary" when accessed through an instance. Functions are **descriptors**.

If a callable should not receive an instance automatically, prefer `@staticmethod` or store a callable object that does not implement binding semantics.


## 2. Problem — Fix accidental binding using `staticmethod`

Rewrite the following class so that `normalize` can be called safely through both the class and an instance:

```python
class TextTools:
    def normalize(text):
        return " ".join(text.lower().split())
```

Required behavior:

```python
TextTools.normalize("  Hello   WORLD  ") == "hello world"
TextTools().normalize("  Hello   WORLD  ") == "hello world"
```


### Solution


In [3]:
class TextTools:
    @staticmethod
    def normalize(text):
        return " ".join(text.lower().split())


assert TextTools.normalize("  Hello   WORLD  ") == "hello world"
assert TextTools().normalize("  Hello   WORLD  ") == "hello world"

print(TextTools.normalize("  Hello   WORLD  "))
print(TextTools().normalize("  Hello   WORLD  "))


hello world
hello world


### Why `staticmethod` is the right tool

Use `staticmethod` when:

- the operation conceptually belongs to the class,
- it does not need `self`,
- it does not need `cls`,
- and you want identical behavior through class and instance access.

Do not add a dummy `self` parameter merely to suppress an error.


## 3. Problem — Compare plain function, `staticmethod`, and `classmethod`

Create a class `Demo` with:

- a plain method `regular`
- a static method `static`
- a class method `klass`

Each should return enough information to reveal what object Python automatically supplies.

Then inspect the raw values stored in `Demo.__dict__`.


### Solution


In [4]:
class Demo:
    label = "DEMO"

    def regular(self):
        return ("regular", self)

    @staticmethod
    def static():
        return ("static", None)

    @classmethod
    def klass(cls):
        return ("klass", cls)


d = Demo()

print("Through class:")
print("Demo.regular ->", Demo.regular)
print("Demo.static  ->", Demo.static)
print("Demo.klass   ->", Demo.klass)

print("\nThrough instance:")
print("d.regular ->", d.regular)
print("d.static  ->", d.static)
print("d.klass   ->", d.klass)

print("\nCalls:")
print(d.regular())
print(d.static())
print(d.klass())

print("\nRaw class namespace values:")
for name in ("regular", "static", "klass"):
    print(name, "=>", Demo.__dict__[name], "| type:", type(Demo.__dict__[name]).__name__)


Through class:
Demo.regular -> <function Demo.regular at 0x0000021A5678CEA0>
Demo.static  -> <function Demo.static at 0x0000021A5678CE00>
Demo.klass   -> <bound method Demo.klass of <class '__main__.Demo'>>

Through instance:
d.regular -> <bound method Demo.regular of <__main__.Demo object at 0x0000021A566EE7B0>>
d.static  -> <function Demo.static at 0x0000021A5678CE00>
d.klass   -> <bound method Demo.klass of <class '__main__.Demo'>>

Calls:
('regular', <__main__.Demo object at 0x0000021A566EE7B0>)
('static', None)
('klass', <class '__main__.Demo'>)

Raw class namespace values:
regular => <function Demo.regular at 0x0000021A5678CEA0> | type: function
static => <staticmethod(<function Demo.static at 0x0000021A5678CE00>)> | type: staticmethod
klass => <classmethod(<function Demo.klass at 0x0000021A5678D080>)> | type: classmethod


### Important observation

Inside `Demo.__dict__`:

- `regular` is a function object.
- `static` is a `staticmethod` descriptor object.
- `klass` is a `classmethod` descriptor object.

Normal attribute access invokes descriptor behavior and may therefore return a different object from the raw namespace entry.


## 4. Problem — Store externally defined functions as class attributes

Define these functions outside a class:

```python
def add(a, b): ...
def multiply(a, b): ...
```

Then create a class `Operations` that exposes them as class attributes.

Your design must support:

```python
Operations.add(2, 3) == 5
Operations.multiply(4, 5) == 20
Operations().add(2, 3) == 5
```

Avoid accidental instance binding.


### Solution


In [5]:
def add(a, b):
    return a + b


def multiply(a, b):
    return a * b


class Operations:
    add = staticmethod(add)
    multiply = staticmethod(multiply)


assert Operations.add(2, 3) == 5
assert Operations.multiply(4, 5) == 20
assert Operations().add(2, 3) == 5

print(Operations.add(2, 3))
print(Operations.multiply(4, 5))
print(Operations().add(2, 3))


5
20
5


### Common pitfall

This version is dangerous if instance access is expected:

```python
class BadOperations:
    add = add
```

Because a normal function stored on a class is a descriptor, `BadOperations().add(2, 3)` tries to inject the instance as another argument.


In [6]:
class BadOperations:
    add = add


print("Class call works:", BadOperations.add(2, 3))

try:
    BadOperations().add(2, 3)
except TypeError as exc:
    print("Instance call fails:", exc)


Class call works: 5
Instance call fails: add() takes 2 positional arguments but 3 were given


## 5. Problem — Build a callable object and store it as a class attribute

Create a callable class `Multiplier`:

```python
double = Multiplier(2)
triple = Multiplier(3)
```

Then create a class `MathPresets` whose class attributes `double` and `triple` are those callable objects.

Required behavior:

```python
MathPresets.double(10) == 20
MathPresets.triple(10) == 30
MathPresets().double(10) == 20
```

Explain why this avoids normal function binding.


### Solution


In [7]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

    def __repr__(self):
        return f"Multiplier(factor={self.factor})"


class MathPresets:
    double = Multiplier(2)
    triple = Multiplier(3)


assert MathPresets.double(10) == 20
assert MathPresets.triple(10) == 30
assert MathPresets().double(10) == 20

print(MathPresets.double)
print(MathPresets.double(10))
print(MathPresets().double(10))


Multiplier(factor=2)
20
20


### Explanation

`Multiplier` instances are callable because they implement `__call__`, but they do not automatically become bound methods.

Normal function objects have special descriptor behavior. A generic callable object does not receive the instance automatically unless its class explicitly implements descriptor behavior such as `__get__`.


## 6. Problem — Use a class-level dispatch table

Create a `CommandProcessor` with a class attribute named `commands`.

It should map command names to callables:

- `"upper"` → uppercase a string
- `"lower"` → lowercase a string
- `"title"` → title-case a string
- `"reverse"` → reverse a string

Add:

```python
@classmethod
def execute(cls, command, text):
    ...
```

Requirements:

- use the class-level registry
- raise a clear `ValueError` for unknown commands
- do not use a long `if`/`elif` chain


### Solution


In [8]:
class CommandProcessor:
    commands = {
        "upper": str.upper,
        "lower": str.lower,
        "title": str.title,
        "reverse": lambda text: text[::-1],
    }

    @classmethod
    def execute(cls, command, text):
        try:
            operation = cls.commands[command]
        except KeyError as exc:
            available = ", ".join(sorted(cls.commands))
            raise ValueError(
                f"Unknown command {command!r}. Available commands: {available}"
            ) from exc

        return operation(text)


assert CommandProcessor.execute("upper", "hello") == "HELLO"
assert CommandProcessor.execute("reverse", "abc") == "cba"

for command in CommandProcessor.commands:
    print(command, "->", CommandProcessor.execute(command, "hello world"))

try:
    CommandProcessor.execute("missing", "hello")
except ValueError as exc:
    print("Expected error:", exc)


upper -> HELLO WORLD
lower -> hello world
title -> Hello World
reverse -> dlrow olleh
Expected error: Unknown command 'missing'. Available commands: lower, reverse, title, upper


### Best practice

Dispatch tables are often cleaner than large conditionals when behavior is naturally keyed by a name.

Benefits include:

- simpler extension,
- easier testing,
- introspection of supported operations,
- fewer control-flow branches.


## 7. Problem — Make the registry inheritance-friendly

Extend the previous idea.

Create:

- `BaseFormatter` with `"upper"` and `"lower"`
- `ExtendedFormatter` that adds `"slug"`

Do **not** mutate the parent class's registry accidentally.

Required property:

```python
"slug" not in BaseFormatter.formats
"slug" in ExtendedFormatter.formats
```


### Solution


In [9]:
class BaseFormatter:
    formats = {
        "upper": str.upper,
        "lower": str.lower,
    }

    @classmethod
    def format(cls, style, text):
        try:
            operation = cls.formats[style]
        except KeyError as exc:
            raise ValueError(f"Unsupported style: {style!r}") from exc
        return operation(text)


class ExtendedFormatter(BaseFormatter):
    formats = {
        **BaseFormatter.formats,
        "slug": lambda text: "-".join(text.lower().split()),
    }


assert "slug" not in BaseFormatter.formats
assert "slug" in ExtendedFormatter.formats
assert ExtendedFormatter.format("slug", "Hello Callable World") == "hello-callable-world"

print(BaseFormatter.formats)
print(ExtendedFormatter.formats)


{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>}
{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>, 'slug': <function ExtendedFormatter.<lambda> at 0x0000021A5678D620>}


### Why copying matters

Class attributes are inherited by reference until overridden.

If a subclass performs:

```python
ExtendedFormatter.formats["slug"] = ...
```

without first replacing `formats`, both classes may still refer to the same dictionary.

For mutable class attributes, explicitly create a new object when subclass-specific mutation is intended.


## 8. Problem — Dynamic callable replacement / monkey patching

Create a class:

```python
class Pipeline:
    transform = ...
```

The default transform should strip whitespace.

Then replace the class attribute at runtime so that it strips whitespace **and** uppercases text.

Make both transformations work through the class.

Finally restore the original callable safely.


### Solution


In [10]:
class Pipeline:
    transform = staticmethod(str.strip)


print("Original:", Pipeline.transform("   hello   "))

original_transform = Pipeline.__dict__["transform"]

try:
    Pipeline.transform = staticmethod(lambda text: text.strip().upper())
    print("Patched:", Pipeline.transform("   hello   "))
    assert Pipeline.transform("   hello   ") == "HELLO"
finally:
    Pipeline.transform = original_transform


print("Restored:", Pipeline.transform("   hello   "))
assert Pipeline.transform("   hello   ") == "hello"


Original: hello
Patched: HELLO
Restored: hello


### Best practice

Runtime patching is useful in controlled contexts such as tests, but can make production code harder to reason about.

If temporary replacement is required:

1. save the original raw class attribute,
2. patch in a narrow scope,
3. restore it in `finally`,
4. preserve descriptor wrappers such as `staticmethod` when necessary.


## 9. Problem — Introspect only callable public class attributes

Write a function:

```python
def public_callables(cls):
    ...
```

It should return the names of public attributes that are callable when accessed through the class.

For example:

```python
class Service:
    version = "1.0"

    @staticmethod
    def ping(): ...

    @classmethod
    def create(cls): ...

    def instance_method(self): ...
```

Expected names should include all callable public attributes exposed by normal class lookup.


### Solution


In [11]:
def public_callables(cls):
    result = []

    for name in dir(cls):
        if name.startswith("_"):
            continue

        value = getattr(cls, name)

        if callable(value):
            result.append(name)

    return result


class Service:
    version = "1.0"

    @staticmethod
    def ping():
        return "pong"

    @classmethod
    def create(cls):
        return cls()

    def instance_method(self):
        return "instance"


print(public_callables(Service))


['create', 'instance_method', 'ping']


### Subtle point

`callable(getattr(cls, name))` checks the **resolved attribute**, not merely the raw object in `cls.__dict__`.

That distinction matters because descriptors can transform attributes during lookup.


## 10. Problem — Inspect raw descriptors without triggering them

Create a class `API` containing:

- a normal method
- a `staticmethod`
- a `classmethod`

Then write code that classifies each raw namespace value without invoking descriptor binding.

Expected classification:

- regular method → function
- static method → staticmethod
- class method → classmethod


### Solution


In [12]:
import types


class API:
    def regular(self):
        return "regular"

    @staticmethod
    def static():
        return "static"

    @classmethod
    def klass(cls):
        return cls.__name__


def classify_raw_attribute(cls, name):
    raw = cls.__dict__[name]

    if isinstance(raw, staticmethod):
        return "staticmethod"
    if isinstance(raw, classmethod):
        return "classmethod"
    if isinstance(raw, types.FunctionType):
        return "function"

    return type(raw).__name__


for name in ("regular", "static", "klass"):
    print(name, "->", classify_raw_attribute(API, name))


regular -> function
static -> staticmethod
klass -> classmethod


## 11. Problem — Build a validated plugin registry

Create a base class `PluginHost` with:

```python
plugins = {}
```

Add a class method:

```python
register(name, plugin)
```

Validation rules:

- `name` must be a non-empty string
- `plugin` must be callable
- duplicate names are rejected unless `replace=True`

Add:

```python
run(name, *args, **kwargs)
```

which executes the registered callable.

Use clear exceptions.


### Solution


In [13]:
class PluginHost:
    plugins = {}

    @classmethod
    def register(cls, name, plugin, *, replace=False):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Plugin name must be a non-empty string.")

        if not callable(plugin):
            raise TypeError("Plugin must be callable.")

        if name in cls.plugins and not replace:
            raise KeyError(f"Plugin {name!r} is already registered.")

        cls.plugins[name] = plugin

    @classmethod
    def run(cls, name, *args, **kwargs):
        try:
            plugin = cls.plugins[name]
        except KeyError as exc:
            raise KeyError(f"No plugin registered under {name!r}.") from exc

        return plugin(*args, **kwargs)


def square(x):
    return x * x


PluginHost.plugins = {}  # isolate this example
PluginHost.register("square", square)
PluginHost.register("sum", lambda *values: sum(values))

assert PluginHost.run("square", 8) == 64
assert PluginHost.run("sum", 1, 2, 3, 4) == 10

print(PluginHost.run("square", 8))
print(PluginHost.run("sum", 1, 2, 3, 4))

try:
    PluginHost.register("square", square)
except KeyError as exc:
    print("Duplicate rejected:", exc)


64
10
Duplicate rejected: "Plugin 'square' is already registered."


## 12. Problem — Decorator-based registration

Improve the registry so callers can register functions using:

```python
@Registry.register("double")
def double(x):
    return x * 2
```

The decorator must return the original function unchanged.


### Solution


In [14]:
class Registry:
    operations = {}

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Registration name must be a non-empty string.")

        def decorator(func):
            if not callable(func):
                raise TypeError("Registered object must be callable.")
            if name in cls.operations:
                raise KeyError(f"Operation {name!r} already exists.")

            cls.operations[name] = func
            return func

        return decorator

    @classmethod
    def execute(cls, name, *args, **kwargs):
        try:
            func = cls.operations[name]
        except KeyError as exc:
            raise ValueError(f"Unknown operation: {name!r}") from exc
        return func(*args, **kwargs)


@Registry.register("double")
def double(x):
    return x * 2


@Registry.register("power")
def power(base, exponent=2):
    return base ** exponent


assert Registry.execute("double", 12) == 24
assert Registry.execute("power", 3, exponent=4) == 81

print(Registry.operations)
print(Registry.execute("double", 12))
print(Registry.execute("power", 3, exponent=4))


{'double': <function double at 0x0000021A5678DE40>, 'power': <function power at 0x0000021A5678E200>}
24
81


## 13. Problem — Preserve metadata in callable decorators

A decorator often replaces one callable with another.

Write a timing-free logging decorator named `trace` that prints the function name and arguments before calling the function.

Use `functools.wraps` so metadata such as `__name__` and `__doc__` is preserved.

Then store the decorated function as a `staticmethod` on a class.


### Solution


In [15]:
from functools import wraps


def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} args={args!r} kwargs={kwargs!r}")
        return func(*args, **kwargs)

    return wrapper


@trace
def clean_text(text):
    """Normalize whitespace and lowercase the text."""
    return " ".join(text.lower().split())


class Cleaner:
    clean = staticmethod(clean_text)


result = Cleaner.clean("  Hello   WORLD  ")

print("Result:", result)
print("Name:", Cleaner.clean.__name__)
print("Doc:", Cleaner.clean.__doc__)

assert result == "hello world"
assert Cleaner.clean.__name__ == "clean_text"


Calling clean_text args=('  Hello   WORLD  ',) kwargs={}
Result: hello world
Name: clean_text
Doc: Normalize whitespace and lowercase the text.


## 14. Problem — Callable attribute shadowing in inheritance

Study the classes below:

```python
class Parent:
    @staticmethod
    def action(x):
        return f"parent:{x}"

class Child(Parent):
    pass
```

Perform these steps:

1. call `Child.action("A")`
2. override `Child.action` with a new callable
3. prove `Parent.action` is unchanged
4. delete the child override
5. prove inherited lookup works again


### Solution


In [16]:
class Parent:
    @staticmethod
    def action(x):
        return f"parent:{x}"


class Child(Parent):
    pass


print("Inherited:", Child.action("A"))

Child.action = staticmethod(lambda x: f"child:{x}")

print("Child override:", Child.action("A"))
print("Parent unchanged:", Parent.action("A"))

assert Child.action("A") == "child:A"
assert Parent.action("A") == "parent:A"

del Child.action

print("Inherited again:", Child.action("A"))
assert Child.action("A") == "parent:A"


Inherited: parent:A
Child override: child:A
Parent unchanged: parent:A
Inherited again: parent:A


### Lookup rule

For `Child.action`, Python searches roughly along the method resolution order (MRO):

1. `Child`
2. base classes in MRO order

Deleting the shadowing attribute reveals the inherited attribute again.


## 15. Problem — Build a custom descriptor that returns different callables

Create a descriptor `EnvironmentAction` that receives two functions:

- one for class access
- one for instance access

Desired behavior:

```python
class Example:
    action = EnvironmentAction(class_func, instance_func)

Example.action()      # uses class_func
Example().action()    # uses instance_func
```

This exercise demonstrates that attribute access itself can manufacture different callable objects.


### Solution


In [17]:
class EnvironmentAction:
    def __init__(self, class_func, instance_func):
        self.class_func = class_func
        self.instance_func = instance_func

    def __get__(self, instance, owner):
        if instance is None:
            return self.class_func

        return lambda *args, **kwargs: self.instance_func(
            instance, *args, **kwargs
        )


def class_behavior():
    return "called through class"


def instance_behavior(instance):
    return f"called through instance of {type(instance).__name__}"


class Example:
    action = EnvironmentAction(class_behavior, instance_behavior)


print(Example.action())
print(Example().action())

assert Example.action() == "called through class"
assert Example().action() == "called through instance of Example"


called through class
called through instance of Example


## 16. Problem — Implement your own simplified `staticmethod`

Implement a descriptor named `MyStaticMethod` with behavior similar to Python's built-in `staticmethod`.

It should return the same underlying function whether accessed through the class or an instance.


### Solution


In [18]:
class MyStaticMethod:
    def __init__(self, func):
        if not callable(func):
            raise TypeError("MyStaticMethod requires a callable.")
        self.func = func

    def __get__(self, instance, owner):
        return self.func


class Calculator:
    @MyStaticMethod
    def add(a, b):
        return a + b


c = Calculator()

assert Calculator.add(2, 3) == 5
assert c.add(2, 3) == 5

print(Calculator.add(2, 3))
print(c.add(2, 3))


5
5


## 17. Problem — Implement your own simplified `classmethod`

Implement `MyClassMethod`.

It should bind the owning class as the first argument regardless of whether access happens through the class or an instance.

Hint: use a closure or `types.MethodType`.


### Solution


In [19]:
from types import MethodType


class MyClassMethod:
    def __init__(self, func):
        if not callable(func):
            raise TypeError("MyClassMethod requires a callable.")
        self.func = func

    def __get__(self, instance, owner):
        return MethodType(self.func, owner)


class Factory:
    kind = "base"

    @MyClassMethod
    def describe(cls):
        return f"{cls.__name__}:{cls.kind}"


class SpecializedFactory(Factory):
    kind = "special"


assert Factory.describe() == "Factory:base"
assert SpecializedFactory.describe() == "SpecializedFactory:special"
assert SpecializedFactory().describe() == "SpecializedFactory:special"

print(Factory.describe())
print(SpecializedFactory.describe())
print(SpecializedFactory().describe())


Factory:base
SpecializedFactory:special
SpecializedFactory:special


## 18. Problem — Signature validation for registered callables

Suppose every transform in a pipeline must accept exactly one required positional argument.

Use `inspect.signature` to reject obviously incompatible callables at registration time.

Accept:

```python
def clean(text): ...
```

Reject:

```python
def combine(a, b): ...
```

For this exercise, restrict accepted functions to those having exactly one required positional parameter and no additional required positional parameters.


### Solution


In [20]:
import inspect


class TransformRegistry:
    transforms = {}

    @staticmethod
    def _validate_signature(func):
        signature = inspect.signature(func)

        required_positional = [
            parameter
            for parameter in signature.parameters.values()
            if parameter.kind
            in (
                inspect.Parameter.POSITIONAL_ONLY,
                inspect.Parameter.POSITIONAL_OR_KEYWORD,
            )
            and parameter.default is inspect.Parameter.empty
        ]

        if len(required_positional) != 1:
            raise TypeError(
                f"{func!r} must have exactly one required positional parameter; "
                f"signature is {signature}"
            )

    @classmethod
    def register(cls, name, func):
        if not callable(func):
            raise TypeError("Transform must be callable.")

        cls._validate_signature(func)
        cls.transforms[name] = func


def clean(text):
    return text.strip()


def combine(a, b):
    return a + b


TransformRegistry.transforms = {}
TransformRegistry.register("clean", clean)

try:
    TransformRegistry.register("combine", combine)
except TypeError as exc:
    print("Rejected:", exc)

print(TransformRegistry.transforms)


Rejected: <function combine at 0x0000021A5678E3E0> must have exactly one required positional parameter; signature is (a, b)
{'clean': <function clean at 0x0000021A5678EB60>}


### Production note

Signature validation can become complicated because Python callables may contain:

- positional-only parameters,
- keyword-only parameters,
- `*args`,
- `**kwargs`,
- bound parameters,
- C-extension callables with limited signature metadata.

Validate only as strictly as your application genuinely requires.


## 19. Problem — A strategy-based processor with callable class attributes

Build an extensible `DataProcessor`.

Requirements:

- class attribute `strategies` maps names to callables
- `register_strategy` validates the callable
- `process(strategy, values)` selects behavior
- built-in strategies:
  - `sum`
  - `max`
  - `min`
  - `mean`
- subclass `SafeDataProcessor` overrides `process` to reject empty input
- no duplicated strategy table in the subclass


### Solution


In [21]:
class DataProcessor:
    strategies = {
        "sum": sum,
        "max": max,
        "min": min,
        "mean": lambda values: sum(values) / len(values),
    }

    @classmethod
    def register_strategy(cls, name, func):
        if not isinstance(name, str) or not name:
            raise ValueError("Strategy name must be a non-empty string.")
        if not callable(func):
            raise TypeError("Strategy must be callable.")

        # Copy-on-write prevents subclass registration from mutating a
        # dictionary inherited directly from a parent class.
        if "strategies" not in cls.__dict__:
            cls.strategies = dict(cls.strategies)

        cls.strategies[name] = func

    @classmethod
    def process(cls, strategy, values):
        try:
            func = cls.strategies[strategy]
        except KeyError as exc:
            raise ValueError(f"Unknown strategy: {strategy!r}") from exc

        return func(values)


class SafeDataProcessor(DataProcessor):
    @classmethod
    def process(cls, strategy, values):
        values = list(values)

        if not values:
            raise ValueError("values must not be empty")

        return super().process(strategy, values)


values = [10, 20, 30, 40]

for name in ("sum", "max", "min", "mean"):
    print(name, "->", SafeDataProcessor.process(name, values))

SafeDataProcessor.register_strategy(
    "range",
    lambda values: max(values) - min(values),
)

print("range ->", SafeDataProcessor.process("range", values))

assert "range" in SafeDataProcessor.strategies
assert "range" not in DataProcessor.strategies


sum -> 100
max -> 40
min -> 10
mean -> 25.0
range -> 30


## 20. Problem — Instance override vs class callable

A class has a callable class attribute:

```python
class Worker:
    task = ...
```

Demonstrate that an instance can shadow a class attribute by assigning another callable to `worker.task`.

Then delete the instance attribute and show that class lookup becomes visible again.

Use a callable object or `staticmethod` so the example focuses on shadowing instead of accidental binding.


### Solution


In [22]:
class Worker:
    task = staticmethod(lambda value: f"class:{value}")


worker = Worker()

print(worker.task("job"))

worker.task = lambda value: f"instance:{value}"
print(worker.task("job"))

assert worker.task("job") == "instance:job"
assert Worker.task("job") == "class:job"

del worker.task

print(worker.task("job"))
assert worker.task("job") == "class:job"


class:job
instance:job
class:job


### Lookup intuition

For normal non-data descriptors and ordinary class attributes, an instance attribute may shadow the class attribute.

This is one reason debugging attribute lookup sometimes requires inspecting both:

```python
instance.__dict__
type(instance).__dict__
```


## 21. Problem — Detect where a callable came from in the MRO

Write:

```python
def defining_class(cls, attribute_name):
    ...
```

It should return the first class in `cls.__mro__` whose raw `__dict__` contains the attribute.

Test it with a three-level hierarchy.


### Solution


In [23]:
def defining_class(cls, attribute_name):
    for base in cls.__mro__:
        if attribute_name in base.__dict__:
            return base
    raise AttributeError(attribute_name)


class A:
    @staticmethod
    def action():
        return "A"


class B(A):
    pass


class C(B):
    pass


print(defining_class(C, "action"))
assert defining_class(C, "action") is A

B.action = staticmethod(lambda: "B")

print(defining_class(C, "action"))
assert defining_class(C, "action") is B
assert C.action() == "B"


<class '__main__.A'>
<class '__main__.B'>


## 22. Problem — Build an immutable callable registry

Mutable class dictionaries are convenient but can be changed accidentally.

Create a registry exposed as a read-only mapping using `types.MappingProxyType`.

The external user must not be able to do:

```python
Registry.operations["x"] = ...
```

But the class itself should still support controlled registration through a method.


### Solution


In [24]:
from types import MappingProxyType


class ImmutableRegistry:
    _operations = {}

    @classmethod
    def register(cls, name, func):
        if not callable(func):
            raise TypeError("func must be callable")
        cls._operations[name] = func

    @classmethod
    def operations(cls):
        return MappingProxyType(cls._operations)


ImmutableRegistry.register("double", lambda x: x * 2)

view = ImmutableRegistry.operations()

print(view)
print(view["double"](5))

try:
    view["triple"] = lambda x: x * 3
except TypeError as exc:
    print("Read-only protection:", exc)


{'double': <function <lambda> at 0x0000021A5678F560>}
10
Read-only protection: 'mappingproxy' object does not support item assignment


## 23. Problem — Advanced dispatch with fallback behavior

Create a class `Serializer` with callable class attributes in a registry:

- `"text"` converts to `str`
- `"repr"` converts using `repr`
- `"csv"` joins an iterable with commas

Its `serialize` method should:

1. use the named serializer if present,
2. optionally use a fallback callable,
3. otherwise raise `ValueError`.

The fallback must also be validated with `callable()`.


### Solution


In [25]:
class Serializer:
    serializers = {
        "text": str,
        "repr": repr,
        "csv": lambda values: ",".join(map(str, values)),
    }

    @classmethod
    def serialize(cls, kind, value, *, fallback=None):
        serializer = cls.serializers.get(kind)

        if serializer is None:
            if fallback is None:
                raise ValueError(f"Unknown serializer: {kind!r}")

            if not callable(fallback):
                raise TypeError("fallback must be callable")

            serializer = fallback

        return serializer(value)


print(Serializer.serialize("text", 123))
print(Serializer.serialize("repr", {"a": 1}))
print(Serializer.serialize("csv", [1, 2, 3]))
print(
    Serializer.serialize(
        "json-ish",
        {"a": 1},
        fallback=lambda value: f"<fallback>{value!r}</fallback>",
    )
)


123
{'a': 1}
1,2,3
<fallback>{'a': 1}</fallback>


## 24. Problem — Debug callable lookup with `inspect.getattr_static`

Normal `getattr` may trigger descriptors.

Use `inspect.getattr_static` to retrieve an attribute without invoking descriptor logic.

Compare:

```python
getattr(Demo, "static")
inspect.getattr_static(Demo, "static")
```

and likewise for a `classmethod`.


### Solution


In [26]:
import inspect


class LookupDemo:
    @staticmethod
    def static():
        return "static"

    @classmethod
    def klass(cls):
        return cls.__name__


for name in ("static", "klass"):
    normal = getattr(LookupDemo, name)
    raw = inspect.getattr_static(LookupDemo, name)

    print(f"{name}:")
    print("  getattr              ->", normal)
    print("  inspect.getattr_static ->", raw)
    print("  raw type             ->", type(raw).__name__)


static:
  getattr              -> <function LookupDemo.static at 0x0000021A5678F9C0>
  inspect.getattr_static -> <staticmethod(<function LookupDemo.static at 0x0000021A5678F9C0>)>
  raw type             -> staticmethod
klass:
  getattr              -> <bound method LookupDemo.klass of <class '__main__.LookupDemo'>>
  inspect.getattr_static -> <classmethod(<function LookupDemo.klass at 0x0000021A5678F740>)>
  raw type             -> classmethod


### Why this is useful

`inspect.getattr_static` is valuable in debuggers, linters, documentation tools, and frameworks because it can inspect raw attributes without executing potentially surprising descriptor logic.


## 25. Capstone Problem — Extensible transformation engine

Design a reusable transformation framework.

### Requirements

Create a class `TransformationEngine` with:

- class-level transformation registry
- decorator-based registration
- duplicate-name protection
- clear errors
- ability to inspect registered transformations
- ability to execute one transformation
- ability to execute a sequence of transformations
- subclass-safe registration using copy-on-write
- callable validation

Register at least these transformations:

- `strip`
- `lower`
- `upper`
- `slug`
- `reverse`

Then create a subclass `CustomEngine` and register:

- `bracket` → wraps the value in `[...]`

Verify that `bracket` does **not** appear in the parent registry.


### Capstone Solution


In [27]:
from types import MappingProxyType


class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(cls._transformations)

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Transformation name must be a non-empty string.")

        def decorator(func):
            if not callable(func):
                raise TypeError("Transformation must be callable.")

            cls._ensure_own_registry()

            if name in cls._transformations:
                raise KeyError(
                    f"Transformation {name!r} is already registered "
                    f"for {cls.__name__}."
                )

            cls._transformations[name] = func
            return func

        return decorator

    @classmethod
    def available(cls):
        return tuple(sorted(cls._transformations))

    @classmethod
    def registry_view(cls):
        return MappingProxyType(cls._transformations)

    @classmethod
    def execute(cls, name, value):
        try:
            transform = cls._transformations[name]
        except KeyError as exc:
            raise ValueError(
                f"Unknown transformation {name!r}. "
                f"Available: {', '.join(cls.available())}"
            ) from exc

        return transform(value)

    @classmethod
    def pipeline(cls, value, *names):
        for name in names:
            value = cls.execute(name, value)
        return value


@TransformationEngine.register("strip")
def strip_transform(text):
    return text.strip()


@TransformationEngine.register("lower")
def lower_transform(text):
    return text.lower()


@TransformationEngine.register("upper")
def upper_transform(text):
    return text.upper()


@TransformationEngine.register("slug")
def slug_transform(text):
    return "-".join(text.split())


@TransformationEngine.register("reverse")
def reverse_transform(text):
    return text[::-1]


class CustomEngine(TransformationEngine):
    pass


@CustomEngine.register("bracket")
def bracket_transform(text):
    return f"[{text}]"


print("Parent:", TransformationEngine.available())
print("Child :", CustomEngine.available())

result = TransformationEngine.pipeline(
    "   Hello Callable World   ",
    "strip",
    "lower",
    "slug",
)

print("Pipeline result:", result)

custom_result = CustomEngine.pipeline(
    "  Hello  ",
    "strip",
    "upper",
    "bracket",
)

print("Custom pipeline result:", custom_result)

assert result == "hello-callable-world"
assert custom_result == "[HELLO]"
assert "bracket" not in TransformationEngine.available()
assert "bracket" in CustomEngine.available()


Parent: ('lower', 'reverse', 'slug', 'strip', 'upper')
Child : ('bracket', 'lower', 'reverse', 'slug', 'strip', 'upper')
Pipeline result: hello-callable-world
Custom pipeline result: [HELLO]


## 26. Capstone Extension — Add aliases safely

Extend the transformation engine with an alias mechanism:

```python
CustomEngine.alias("shout", "upper")
```

Requirements:

- alias target must already exist
- alias name must not already exist
- alias should reference the same callable object
- errors should be explicit


### Solution


In [28]:
class AliasEngine(CustomEngine):
    @classmethod
    def alias(cls, alias_name, target_name):
        cls._ensure_own_registry()

        if target_name not in cls._transformations:
            raise KeyError(f"Unknown target transformation: {target_name!r}")

        if alias_name in cls._transformations:
            raise KeyError(f"Alias name already exists: {alias_name!r}")

        cls._transformations[alias_name] = cls._transformations[target_name]


AliasEngine.alias("shout", "upper")

assert AliasEngine.execute("shout", "hello") == "HELLO"
assert (
    AliasEngine.registry_view()["shout"]
    is AliasEngine.registry_view()["upper"]
)

print(AliasEngine.execute("shout", "hello"))
print("Same callable:",
      AliasEngine.registry_view()["shout"]
      is AliasEngine.registry_view()["upper"])


HELLO
Same callable: True


## 27. Challenge — Explain this surprising behavior

Predict the output and explain it:

```python
def external(x):
    return x

class Mystery:
    a = external
    b = staticmethod(external)

m = Mystery()

print(Mystery.a)
print(Mystery.b)
print(m.a)
print(m.b)
```

Then test whether these calls work:

```python
Mystery.a(10)
Mystery.b(10)
m.a(10)
m.b(10)
```


### Solution


In [29]:
def external(x):
    return x


class Mystery:
    a = external
    b = staticmethod(external)


m = Mystery()

print("Mystery.a:", Mystery.a)
print("Mystery.b:", Mystery.b)
print("m.a:", m.a)
print("m.b:", m.b)

print("\nCalls:")
print("Mystery.a(10):", Mystery.a(10))
print("Mystery.b(10):", Mystery.b(10))

try:
    print("m.a(10):", m.a(10))
except TypeError as exc:
    print("m.a(10) failed:", exc)

print("m.b(10):", m.b(10))


Mystery.a: <function external at 0x0000021A5678FCE0>
Mystery.b: <function external at 0x0000021A5678FCE0>
m.a: <bound method external of <__main__.Mystery object at 0x0000021A566EF380>>
m.b: <function external at 0x0000021A5678FCE0>

Calls:
Mystery.a(10): 10
Mystery.b(10): 10
m.a(10) failed: external() takes 1 positional argument but 2 were given
m.b(10): 10


### Explanation

`a` stores a normal function. Normal functions implement `__get__`, so instance access creates a bound method and injects `m`.

`b` stores a `staticmethod` descriptor, whose job is effectively to suppress that binding and return the original callable.


## 28. Challenge — Callable class attributes vs properties

A `property` may return a callable, but the property itself is not normally called like a method.

Create a class where:

```python
obj.operation
```

returns a callable based on instance state.

Then call:

```python
obj.operation(10)
```

Use this to distinguish:

- a callable stored directly as a class attribute,
- a descriptor that **returns** a callable.


### Solution


In [30]:
class ConfigurableOperation:
    def __init__(self, mode):
        self.mode = mode

    @property
    def operation(self):
        if self.mode == "double":
            return lambda x: x * 2

        if self.mode == "square":
            return lambda x: x * x

        raise ValueError(f"Unsupported mode: {self.mode!r}")


double = ConfigurableOperation("double")
square = ConfigurableOperation("square")

assert double.operation(10) == 20
assert square.operation(10) == 100

print(double.operation(10))
print(square.operation(10))


20
100


## 29. Best-practice checklist

When using callable class attributes:

1. **Choose binding semantics intentionally**
   - instance behavior → normal method
   - class-aware behavior → `classmethod`
   - no automatic binding → `staticmethod`

2. **Validate dynamic callables**
   - use `callable(...)`
   - optionally inspect signatures when your API requires a contract

3. **Be careful with mutable class-level registries**
   - subclasses inherit the same object until they override it
   - use copy-on-write or explicitly copy dictionaries

4. **Use clear error messages**
   - identify the missing/invalid callable name
   - expose available options when useful

5. **Preserve metadata in decorators**
   - use `functools.wraps`

6. **Inspect raw attributes when debugging descriptors**
   - `cls.__dict__[name]`
   - `inspect.getattr_static(cls, name)`

7. **Prefer controlled registration APIs**
   - instead of exposing mutable registry internals directly

8. **Avoid unnecessary monkey patching**
   - if used in tests, restore state reliably

9. **Understand that `callable` and descriptor behavior are separate concepts**
   - a callable object need not bind
   - a descriptor may return a callable

10. **Test class access and instance access separately**
    - they can produce different objects and different call signatures.


## 30. Final review questions

Try answering these without running code first.

### Question 1
Why can a plain function assigned to a class attribute behave differently through an instance?

### Question 2
When should `staticmethod` be preferred over a normal instance method?

### Question 3
Why can a mutable class-level registry cause subclass bugs?

### Question 4
What is the difference between:

```python
getattr(MyClass, "x")
MyClass.__dict__["x"]
```

### Question 5
Can an object be callable without being a function?

### Question 6
Can a non-callable descriptor return a callable?

### Question 7
Why might `inspect.getattr_static` be preferable in framework or debugging code?


## 31. Review answers

1. A normal function implements descriptor behavior. Instance access can bind the instance automatically and create a bound method.

2. Use `staticmethod` when the operation belongs logically to the class namespace but needs neither instance state nor class state.

3. A subclass may inherit the same mutable dictionary object. Mutating it can accidentally modify behavior visible through the parent and sibling subclasses.

4. `getattr` performs normal attribute resolution and descriptor logic. Direct access to `__dict__` retrieves the raw object stored specifically in that namespace.

5. Yes. Any object whose class implements `__call__` may be callable.

6. Yes. For example, a `property` can return a function, lambda, bound method, or callable object.

7. `inspect.getattr_static` avoids triggering descriptor execution, making introspection safer and more faithful to the raw class structure.


## 32. Extra practice ideas

Extend the notebook yourself by implementing:

- registry priorities
- registry namespaces
- unregister support
- aliases with deprecation warnings
- asynchronous callables
- signature normalization
- callable pipelines with exception recovery
- cached callable descriptors
- lazy callable resolution
- callable dependency injection
- metaclass-driven registration
- abstract callable contracts with `abc`
- protocols for callable typing
- generic callable registries using `typing.Callable`
- unit tests using `pytest`
